In [1]:
!gcloud auth application-default login

Your browser has been opened to visit:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=764086051850-6qr4p6gpi6hn506pt8ejuq83di341hur.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8085%2F&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login&state=92khwQPyiQKMaqY99NssVgMQEroa8B&access_type=offline&code_challenge=WpgUb-i0O9guH0R15FuMWxIO-nrkIgkn9jucwrbf1CY&code_challenge_method=S256


Credentials saved to file: [/Users/yt4/.config/gcloud/application_default_credentials.json]

These credentials will be used by any library that requests Application Default Credentials (ADC).

Quota project "open-targets-genetics-dev" was added to ADC which can be used by Google client libraries for billing and quota. Note that some services may still bill the project owning the resource.


In [2]:
import os
import hail as hl
import numpy as np
import scipy as sc
import pandas as pd
import pyspark.sql.functions as f
from pyspark.sql import DataFrame, Window

from gentropy.common.session import Session
from gentropy.dataset.study_index import StudyIndex
from gentropy.dataset.study_locus import StudyLocus
from gentropy.dataset.summary_statistics import SummaryStatistics
from gentropy.datasource.gnomad.ld import GnomADLDMatrix
from gentropy.method.susie_inf import SUSIE_inf
from scipy.stats import chi2
from gentropy.common.spark_helpers import neglog_pvalue_to_mantissa_and_exponent
from gentropy.susie_finemapper import SusieFineMapperStep
from pyspark.sql.functions import explode, col, when
from pyspark.sql.functions import array_contains

from gentropy.finemapping_simulations import FineMappingSimulations

from pyspark.sql.types import (
    DoubleType,
    IntegerType,
    StringType,
    StructField,
    StructType,
)

Loading BokehJS ...

In [3]:
hail_dir = os.path.dirname(hl.__file__)
session = Session(hail_home=hail_dir, start_hail=True, extended_spark_conf={"spark.driver.memory": "12g","spark.kryoserializer.buffer.max": "500m","spark.driver.maxResultSize":"2g"})

24/09/12 12:29:51 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


24/09/12 12:29:52 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


pip-installed Hail requires additional configuration options in Spark referring
  to the path to the Hail Python module directory HAIL_DIR,
  e.g. /path/to/python/site-packages/hail:
    spark.jars=HAIL_DIR/backend/hail-all-spark.jar
    spark.driver.extraClassPath=HAIL_DIR/backend/hail-all-spark.jar
    spark.executor.extraClassPath=./hail-all-spark.jarRunning on Apache Spark version 3.3.4
SparkUI available at http://mib118093s.internal.sanger.ac.uk:4041
Welcome to
     __  __     <>__
    / /_/ /__  __/ /
   / __  / _ `/ / /
  /_/ /_/\_,_/_/_/   version 0.2.127-bb535cd096c5
LOGGING: writing to /dev/null


In [4]:
session.spark.read.parquet

<bound method DataFrameReader.parquet of <pyspark.sql.readwriter.DataFrameReader object at 0x176884340>>

In [3]:
si=StudyIndex.from_parquet(session,"gs://genetics_etl_python_playground/releases/24.06/study_index/gwas_catalog")

In [4]:
sl=StudyLocus.from_parquet(session,"gs://genetics-portal-dev-analysis/dc16/output/gwas_cat_clumped_collected.parquet")
df=sl.df

ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/Users/yt4/Library/Caches/pypoetry/virtualenvs/gentropy-krNFZEZg-py3.10/lib/python3.10/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/Users/yt4/Library/Caches/pypoetry/virtualenvs/gentropy-krNFZEZg-py3.10/lib/python3.10/site-packages/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
  File "/Users/yt4/.pyenv/versions/3.10.8/lib/python3.10/socket.py", line 705, in readinto
    return self._sock.recv_into(b)
KeyboardInterrupt


KeyboardInterrupt: 

In [ ]:
df.printSchema()

root
 |-- variantId: string (nullable = true)
 |-- chromosome: string (nullable = true)
 |-- position: integer (nullable = true)
 |-- region: string (nullable = true)
 |-- studyId: string (nullable = true)
 |-- beta: double (nullable = true)
 |-- zScore: double (nullable = true)
 |-- pValueMantissa: float (nullable = true)
 |-- pValueExponent: integer (nullable = true)
 |-- effectAlleleFrequencyFromSource: float (nullable = true)
 |-- standardError: double (nullable = true)
 |-- subStudyDescription: string (nullable = true)
 |-- qualityControls: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- finemappingMethod: string (nullable = true)
 |-- credibleSetIndex: integer (nullable = true)
 |-- credibleSetlog10BF: double (nullable = true)
 |-- purityMeanR2: double (nullable = true)
 |-- purityMinR2: double (nullable = true)
 |-- locusStart: integer (nullable = true)
 |-- locusEnd: integer (nullable = true)
 |-- sampleSize: integer (nullable = true)
 |-- ldSet: ar

In [ ]:
list_of_sl=["-1001937030930569227",
"-1010313196581703077",
"-1011231071050974680",
"-1012754541153341722",
"-1018262251190305220",
"-1025465871319241012",
"-1028210081609642911",
"-1030347428451315180",
"-103533209966396235",
"-1035576011340159380",
"-1035767924310985877",
"-1039176771908555670",
"-1041429612550728944"]

In [ ]:
df1=df.filter(f.col("studyLocusId").isin(list_of_sl))
#df1=df.filter(f.col("studyLocusId")=="-1001937030930569227")
df1.count()

13

In [ ]:
df1.write.parquet("temp_fiels/nan_locus2")

In [4]:
df2=StudyLocus.from_parquet(session,"temp_fiels/nan_locus2")
df2.df.show()

+--------------------+---------------+----------+--------+------+------------+----------+------+--------------+--------------+-------------------------------+-------------+-------------------+---------------+-----------------+----------------+------------------+------------+-----------+----------+---------+----------+-----+--------------------+
|        studyLocusId|      variantId|chromosome|position|region|     studyId|      beta|zScore|pValueMantissa|pValueExponent|effectAlleleFrequencyFromSource|standardError|subStudyDescription|qualityControls|finemappingMethod|credibleSetIndex|credibleSetlog10BF|purityMeanR2|purityMinR2|locusStart| locusEnd|sampleSize|ldSet|               locus|
+--------------------+---------------+----------+--------+------+------------+----------+------+--------------+--------------+-------------------------------+-------------+-------------------+---------------+-----------------+----------------+------------------+------------+-----------+----------+--------

In [11]:
rows = df2.df.collect()

In [ ]:
results_logging = []
for row in rows:
    result = SusieFineMapperStep.susie_finemapper_one_sl_row_v4_ss_gathered_boundaries(
    session=session,
    study_locus_row=row,
    study_index=si,
    max_causal_snps=10,
    primary_signal_pval_threshold=1,
    secondary_signal_pval_threshold=1,
    purity_mean_r2_threshold=0,
    purity_min_r2_threshold=0,
    cs_lbf_thr=2,
    sum_pips=0.99,
    susie_est_tausq=False,
    run_carma=False,
    carma_tau=0.15,
    run_sumstat_imputation=False,
    carma_time_limit=6000,
    imputed_r2_threshold=0.8,
    ld_score_threshold=5,
    )
    results_logging.append(result)

In [ ]:
len(results_logging)

13

In [ ]:
results_logging[2]["study_locus"].df.show()

+----------------+--------------------+------------+--------------------+--------------------+---------------+----------+--------+-----------------+------------------+-------------------+-----------+-------------------+--------------+--------------+
|credibleSetIndex|        studyLocusId|     studyId|              region|               locus|      variantId|chromosome|position|finemappingMethod|credibleSetlog10BF|       purityMeanR2|purityMinR2|             zScore|pValueMantissa|pValueExponent|
+----------------+--------------------+------------+--------------------+--------------------+---------------+----------+--------+-----------------+------------------+-------------------+-----------+-------------------+--------------+--------------+
|               3|-2211316325451113066|GCST90025982|11:46125702-47625702|[{11_47565900_C_T...|11_47565900_C_T|        11|47565900|        SuSiE-inf| 4.238975060107001|0.08418389634350017|        0.0| -6.084134352541226|      8.538079|            -9|


In [ ]:
results_logging[2]["log"]

,N_gwas_before_dedupl,N_gwas,N_ld,N_overlap,N_outliers,N_imputed,N_final_to_fm,elapsed_time
0,438,438,438,438,0,0,438,24.342805


In [ ]:
results_logging = []
for row in rows[8:]:
    result = SusieFineMapperStep.susie_finemapper_one_sl_row_v4_ss_gathered_boundaries(
    session=session,
    study_locus_row=row,
    study_index=si,
    max_causal_snps=10,
    primary_signal_pval_threshold=1,
    secondary_signal_pval_threshold=1,
    purity_mean_r2_threshold=0,
    purity_min_r2_threshold=0,
    cs_lbf_thr=2,
    sum_pips=0.99,
    susie_est_tausq=False,
    run_carma=True,
    carma_tau=0.3,
    run_sumstat_imputation=False,
    carma_time_limit=60000,
    imputed_r2_threshold=0.8,
    ld_score_threshold=5,
    )
    results_logging.append(result)

In [ ]:
results_logging

[{'study_locus': StudyLocus(_df=DataFrame[credibleSetIndex: int, studyLocusId: bigint, studyId: string, region: string, locus: array<struct<variantId:string,posteriorProbability:double,logBF:double,beta:double>>, variantId: string, chromosome: string, position: int, finemappingMethod: string, credibleSetlog10BF: double, purityMeanR2: double, purityMinR2: double, zScore: double, pValueMantissa: float, pValueExponent: int], _schema=StructType([StructField('studyLocusId', LongType(), False), StructField('variantId', StringType(), False), StructField('chromosome', StringType(), True), StructField('position', IntegerType(), True), StructField('region', StringType(), True), StructField('studyId', StringType(), False), StructField('beta', DoubleType(), True), StructField('zScore', DoubleType(), True), StructField('pValueMantissa', FloatType(), True), StructField('pValueExponent', IntegerType(), True), StructField('effectAlleleFrequencyFromSource', FloatType(), True), StructField('standardErro

In [14]:
np.outer([1,-1,1,-1],[1,-1,1,-1])

array([[ 1, -1,  1, -1],
       [-1,  1, -1,  1],
       [ 1, -1,  1, -1],
       [-1,  1, -1,  1]])

In [8]:
result_logging = SusieFineMapperStep.susie_finemapper_one_sl_row_v4_ss_gathered_boundaries_ldinterface(
session=session,
study_locus_row=df2.df.first(),
study_index=si,
max_causal_snps=10,
primary_signal_pval_threshold=1,
secondary_signal_pval_threshold=1,
purity_mean_r2_threshold=0,
purity_min_r2_threshold=0,
cs_lbf_thr=2,
sum_pips=0.99,
susie_est_tausq=False,
run_carma=False,
carma_tau=0.15,
run_sumstat_imputation=True,
carma_time_limit=60000,
imputed_r2_threshold=0.8,
ld_score_threshold=5,
)

In [9]:
result_logging["log"]

,N_gwas_before_dedupl,N_gwas,N_ld,N_overlap,N_outliers,N_imputed,N_final_to_fm,elapsed_time
0,18626,18626,5031,3087,0,324,3411,115.895348


In [10]:
result_logging["study_locus"].df.show()

+----------------+--------------------+------------+--------------------+--------------------+---------------+----------+--------+-----------------+------------------+------------+-----------+-------------------+--------------+--------------+
|credibleSetIndex|        studyLocusId|     studyId|              region|               locus|      variantId|chromosome|position|finemappingMethod|credibleSetlog10BF|purityMeanR2|purityMinR2|             zScore|pValueMantissa|pValueExponent|
+----------------+--------------------+------------+--------------------+--------------------+---------------+----------+--------+-----------------+------------------+------------+-----------+-------------------+--------------+--------------+
|               6|-6085008223147579656|GCST90002322|15:42236528-43736528|[{15_43658501_A_T...|15_43658501_A_T|        15|43658501|        SuSiE-inf| 50.74062275569449|         1.0|        1.0|-10.207749251850686|     5.4626327|           -24|
|               7|-602229812

In [14]:
import numpy as np
import pandas as pd
import pyspark.sql.functions as f
import scipy as sc
from pyspark.sql import DataFrame, Row, Window
from pyspark.sql.functions import row_number
from pyspark.sql.types import (
    DoubleType,
    IntegerType,
    StringType,
    StructField,
    StructType,
)

from gentropy.common.session import Session
from gentropy.common.spark_helpers import (
    neglog_pvalue_to_mantissa_and_exponent,
    order_array_of_structs_by_field,
)
from gentropy.dataset.study_index import StudyIndex
from gentropy.dataset.study_locus import StudyLocus
from gentropy.datasource.gnomad.ld import GnomADLDMatrix
from gentropy.method.carma import CARMA
from gentropy.method.sumstat_imputation import SummaryStatisticsImputation
from gentropy.method.susie_inf import SUSIE_inf

In [15]:
session=session
study_locus_row=df2.df.first()
study_index=si
max_causal_snps=10
primary_signal_pval_threshold=1
secondary_signal_pval_threshold=1
purity_mean_r2_threshold=0
purity_min_r2_threshold=0
cs_lbf_thr=2
sum_pips=0.99
susie_est_tausq=False
run_carma=True
run_sumstat_imputation=False
carma_time_limit=60000
imputed_r2_threshold=0.8
ld_score_threshold=5

In [16]:
# PLEASE DO NOT REMOVE THIS LINE
pd.DataFrame.iteritems = pd.DataFrame.items

chromosome = study_locus_row["chromosome"]
studyId = study_locus_row["studyId"]
locusStart = study_locus_row["locusStart"]
locusEnd = study_locus_row["locusEnd"]

study_index_df = study_index._df
study_index_df = study_index_df.filter(f.col("studyId") == studyId)
major_population = study_index_df.select(
    "studyId",
    order_array_of_structs_by_field(
        "ldPopulationStructure", "relativeSampleSize"
    )[0]["ldPopulation"].alias("majorPopulation"),
).collect()[0]["majorPopulation"]

region = chromosome + ":" + str(int(locusStart)) + "-" + str(int(locusEnd))

schema = StudyLocus.get_schema()
gwas_df = session.spark.createDataFrame([study_locus_row], schema=schema)
exploded_df = gwas_df.select(f.explode("locus").alias("locus"))

result_df = exploded_df.select(
    "locus.variantId", "locus.beta", "locus.standardError"
)
gwas_df = (
    result_df.withColumn("z", f.col("beta") / f.col("standardError"))
    .withColumn(
        "chromosome", f.split(f.col("variantId"), "_")[0].cast("string")
    )
    .withColumn("position", f.split(f.col("variantId"), "_")[1].cast("int"))
    .filter(f.col("chromosome") == chromosome)
    .filter(f.col("position") >= int(locusStart))
    .filter(f.col("position") <= int(locusEnd))
    .filter(f.col("z").isNotNull())
)

# Remove ALL duplicated variants from GWAS DataFrame - we don't know which is correct
variant_counts = gwas_df.groupBy("variantId").count()
unique_variants = variant_counts.filter(f.col("count") == 1)
gwas_df = gwas_df.join(unique_variants, on="variantId", how="left_semi")

ld_index = (
    GnomADLDMatrix()
    .get_locus_index_boundaries(
        study_locus_row=study_locus_row,
        major_population=major_population,
    )
    .withColumn(
        "variantId",
        f.concat(
            f.lit(chromosome),
            f.lit("_"),
            f.col("`locus.position`"),
            f.lit("_"),
            f.col("alleles").getItem(0),
            f.lit("_"),
            f.col("alleles").getItem(1),
        ).cast("string"),
    )
)
# Remove ALL duplicated variants from ld_index DataFrame - we don't know which is correct
variant_counts = ld_index.groupBy("variantId").count()
unique_variants = variant_counts.filter(f.col("count") == 1)
ld_index = ld_index.join(unique_variants, on="variantId", how="left_semi").sort(
    "idx"
)

2024-09-11 12:32:58.194 Hail: INFO: Table.join: renamed the following fields on the right to avoid name conflicts:
    'faf_index_dict' -> 'faf_index_dict_1'
    'age_distribution' -> 'age_distribution_1'
    'freq_index_dict' -> 'freq_index_dict_1'
    'rf' -> 'rf_1'
    'freq_meta' -> 'freq_meta_1'
    'age_index_dict' -> 'age_index_dict_1'
    'popmax_index_dict' -> 'popmax_index_dict_1'
2024-09-11 12:33:12.534 Hail: INFO: Coerced sorted dataset
2024-09-11 12:33:25.844 Hail: INFO: Coerced sorted dataset
2024-09-11 12:33:40.582 Hail: INFO: Table.join: renamed the following fields on the right to avoid name conflicts:
    'faf_index_dict' -> 'faf_index_dict_1'
    'age_distribution' -> 'age_distribution_1'
    'freq_index_dict' -> 'freq_index_dict_1'
    'rf' -> 'rf_1'
    'freq_meta' -> 'freq_meta_1'
    'age_index_dict' -> 'age_index_dict_1'
    'popmax_index_dict' -> 'popmax_index_dict_1'
2024-09-11 12:33:54.842 Hail: INFO: Coerced sorted dataset
2024-09-11 12:34:08.515 Hail: INFO:

In [17]:
ld_index = (
GnomADLDMatrix()
.get_locus_index_boundaries(
study_locus_row=study_locus_row,
major_population=major_population,
))

In [9]:
ld_index.columns

NameError: name 'ld_index' is not defined

In [18]:
ld_index.show(2)

+---------------------+-----------------------+-------+------------+--------------+-----------+--------------------+-----------+-------------------------+--------+
|original_locus.contig|original_locus.position|alleles|locus.contig|locus.position|pop_freq.AC|         pop_freq.AF|pop_freq.AN|pop_freq.homozygote_count|     idx|
+---------------------+-----------------------+-------+------------+--------------+-----------+--------------------+-----------+-------------------------+--------+
|                   15|               42528739| [T, C]|       chr15|      42236541|        407|0.026387448132780083|      15424|                        4|11167902|
|                   15|               42528982| [T, C]|       chr15|      42236784|        793| 0.05139338950097213|      15430|                       17|11167903|
+---------------------+-----------------------+-------+------------+--------------+-----------+--------------------+-----------+-------------------------+--------+
only showing top

In [ ]:
if not run_sumstat_imputation:
    # Filtering out the variants that are not in the LD matrix, we don't need them
    gwas_index = gwas_df.join(
        ld_index.select("variantId", "alleles", "idx"), on="variantId"
    ).sort("idx")
    gwas_df = gwas_index.select(
        "variantId",
        "z",
        "chromosome",
        "position",
        "beta",
        "StandardError",
    )
    gwas_index = gwas_index.drop(
        "z", "chromosome", "position", "beta", "StandardError"
    )
    if gwas_index.rdd.isEmpty():
        logging.warning("No overlapping variants in the LD Index")
        #return None
    gnomad_ld = GnomADLDMatrix.get_numpy_matrix(
        gwas_index, gnomad_ancestry=major_population
    )

    # Module to remove NANs from the LD matrix
    if sum(sum(np.isnan(gnomad_ld))) > 0:
        gwas_index = gwas_index.toPandas()

        # First round of filtering out the variants with NANs
        nan_count = 1 - (sum(np.isnan(gnomad_ld)) / len(gnomad_ld))
        indices = np.where(nan_count >= 0.98)
        indices = indices[0]
        gnomad_ld = gnomad_ld[indices][:, indices]

        gwas_index = gwas_index.iloc[indices, :]

        if len(gwas_index) == 0:
            logging.warning("No overlapping variants in the LD Index")
            #return None

        # Second round of filtering out the variants with NANs
        nan_count = sum(np.isnan(gnomad_ld))
        indices = np.where(nan_count == 0)
        indices = indices[0]

        gnomad_ld = gnomad_ld[indices][:, indices]
        gwas_index = gwas_index.iloc[indices, :]

        if len(gwas_index) == 0:
            logging.warning("No overlapping variants in the LD Index")
            #return None

        gwas_index = session.spark.createDataFrame(gwas_index)
else:
    gwas_index = gwas_df.join(
        ld_index.select("variantId", "alleles", "idx"), on="variantId"
    ).sort("idx")
    if gwas_index.rdd.isEmpty():
        logging.warning("No overlapping variants in the LD Index")
        #return None
    gwas_index = ld_index
    gnomad_ld = GnomADLDMatrix.get_numpy_matrix(
        gwas_index, gnomad_ancestry=major_population
    )

    # Module to remove NANs from the LD matrix
    if sum(sum(np.isnan(gnomad_ld))) > 0:
        gwas_index = gwas_index.toPandas()

        # First round of filtering out the variants with NANs
        nan_count = 1 - (sum(np.isnan(gnomad_ld)) / len(gnomad_ld))
        indices = np.where(nan_count >= 0.98)
        indices = indices[0]
        gnomad_ld = gnomad_ld[indices][:, indices]

        gwas_index = gwas_index.iloc[indices, :]

        if len(gwas_index) == 0:
            logging.warning("No overlapping variants in the LD Index")
            #return None

        # Second round of filtering out the variants with NANs
        nan_count = sum(np.isnan(gnomad_ld))
        indices = np.where(nan_count == 0)
        indices = indices[0]

        gnomad_ld = gnomad_ld[indices][:, indices]
        gwas_index = gwas_index.iloc[indices, :]

        if len(gwas_index) == 0:
            logging.warning("No overlapping variants in the LD Index")
            #return None

        gwas_index = session.spark.createDataFrame(gwas_index)

# sanity filters on LD matrix
np.fill_diagonal(gnomad_ld, 1)
gnomad_ld[gnomad_ld > 1] = 1
gnomad_ld[gnomad_ld < -1] = -1
upper_triangle = np.triu(gnomad_ld)
gnomad_ld = (
    upper_triangle + upper_triangle.T - np.diag(upper_triangle.diagonal())
)
np.fill_diagonal(gnomad_ld, 1)

In [ ]:
out = SusieFineMapperStep.susie_finemapper_from_prepared_dataframes(
    GWAS_df=gwas_df,
    ld_index=gwas_index,
    gnomad_ld=gnomad_ld,
    L=max_causal_snps,
    session=session,
    studyId=studyId,
    region=region,
    susie_est_tausq=susie_est_tausq,
    run_carma=False,
    run_sumstat_imputation=run_sumstat_imputation,
    carma_time_limit=carma_time_limit,
    imputed_r2_threshold=imputed_r2_threshold,
    ld_score_threshold=ld_score_threshold,
    sum_pips=sum_pips,
    primary_signal_pval_threshold=primary_signal_pval_threshold,
    secondary_signal_pval_threshold=secondary_signal_pval_threshold,
    purity_mean_r2_threshold=purity_mean_r2_threshold,
    purity_min_r2_threshold=purity_min_r2_threshold,
    cs_lbf_thr=cs_lbf_thr,
)

In [ ]:
out

{'study_locus': StudyLocus(_df=DataFrame[credibleSetIndex: int, studyLocusId: bigint, studyId: string, region: string, locus: array<struct<variantId:string,posteriorProbability:double,logBF:double,beta:double>>, variantId: string, chromosome: string, position: int, finemappingMethod: string, credibleSetlog10BF: double, purityMeanR2: double, purityMinR2: double, zScore: double, pValueMantissa: float, pValueExponent: int], _schema=StructType([StructField('studyLocusId', LongType(), False), StructField('variantId', StringType(), False), StructField('chromosome', StringType(), True), StructField('position', IntegerType(), True), StructField('region', StringType(), True), StructField('studyId', StringType(), False), StructField('beta', DoubleType(), True), StructField('zScore', DoubleType(), True), StructField('pValueMantissa', FloatType(), True), StructField('pValueExponent', IntegerType(), True), StructField('effectAlleleFrequencyFromSource', FloatType(), True), StructField('standardError

In [ ]:
out = SusieFineMapperStep.susie_finemapper_from_prepared_dataframes(
    GWAS_df=gwas_df,
    ld_index=gwas_index,
    gnomad_ld=gnomad_ld,
    L=max_causal_snps,
    session=session,
    studyId=studyId,
    region=region,
    susie_est_tausq=susie_est_tausq,
    run_carma=True,
    run_sumstat_imputation=run_sumstat_imputation,
    carma_time_limit=carma_time_limit,
    imputed_r2_threshold=imputed_r2_threshold,
    ld_score_threshold=ld_score_threshold,
    sum_pips=sum_pips,
    primary_signal_pval_threshold=primary_signal_pval_threshold,
    secondary_signal_pval_threshold=secondary_signal_pval_threshold,
    purity_mean_r2_threshold=purity_mean_r2_threshold,
    purity_min_r2_threshold=purity_min_r2_threshold,
    cs_lbf_thr=cs_lbf_thr,
)

/Users/yt4/Projects/gentropy/src/gentropy/method/carma.py:165: RuntimeWarning:

invalid value encountered in log



ValueError: probabilities contain NaN

In [ ]:
GWAS_df=gwas_df
ld_index=gwas_index
gnomad_ld=gnomad_ld
L=max_causal_snps

In [ ]:
# PLEASE DO NOT REMOVE THIS LINE
pd.DataFrame.iteritems = pd.DataFrame.items

start_time = time.time()
GWAS_df = GWAS_df.toPandas()
N_gwas_before_dedupl = len(GWAS_df)

GWAS_df = GWAS_df.drop_duplicates(subset="variantId", keep=False)
GWAS_df = GWAS_df.reset_index()

ld_index = ld_index.toPandas()
ld_index = ld_index.reset_index()

N_gwas = len(GWAS_df)
N_ld = len(ld_index)

# Filtering out the variants that are not in the LD matrix, we don't need them
df_columns = ["variantId", "z"]
GWAS_df = GWAS_df.merge(ld_index, on="variantId", how="inner")
GWAS_df = GWAS_df[df_columns].reset_index()
N_after_merge = len(GWAS_df)

merged_df = GWAS_df.merge(
    ld_index, left_on="variantId", right_on="variantId", how="inner"
)
indices = merged_df["index_y"].values

ld_to_fm = gnomad_ld[indices][:, indices]
z_to_fm = GWAS_df["z"].values

In [ ]:
np.max(z_to_fm)

16.89826739364388

In [ ]:
carma_output = CARMA.time_limited_CARMA_spike_slab_noEM(
    z=z_to_fm, ld=ld_to_fm, sec_threshold=carma_time_limit
)

/Users/yt4/Projects/gentropy/src/gentropy/method/carma.py:165: RuntimeWarning:

invalid value encountered in log



ValueError: probabilities contain NaN

In [ ]:
z=z_to_fm
ld=ld_to_fm

lambda_val=1
Max_Model_Dim = 200_000
all_iter = 1
all_inner_iter = 10
epsilon_threshold = 1e-5
num_causal = 10
tau  = 0.04
outlier_switch = True
outlier_BF_index = 1 / 3.2

In [ ]:
p_snp = len(z)
epsilon_list = epsilon_threshold * p_snp
all_epsilon_threshold = epsilon_threshold * p_snp

# Zero step
all_C_list = CARMA._MCS_modified(
    z=z,
    ld_matrix=ld,
    epsilon=epsilon_list,
    Max_Model_Dim=Max_Model_Dim,
    lambda_val=lambda_val,
    outlier_switch=outlier_switch,
    tau=tau,
    num_causal=num_causal,
    inner_all_iter=all_inner_iter,
    outlier_BF_index=outlier_BF_index,
)


/Users/yt4/Projects/gentropy/src/gentropy/method/carma.py:165: RuntimeWarning:

invalid value encountered in log



ValueError: probabilities contain NaN

In [ ]:
z=z
ld_matrix=ld
epsilon=epsilon_list
Max_Model_Dim=Max_Model_Dim
lambda_val=lambda_val
outlier_switch=outlier_switch
tau=tau
num_causal=num_causal
inner_all_iter=all_inner_iter
outlier_BF_index=outlier_BF_index
input_conditional_S_list=None


In [ ]:
import concurrent.futures
import warnings
from itertools import combinations
from math import floor, lgamma
from typing import Any

import numpy as np
import pandas as pd
from scipy.linalg import det, inv, pinv
from scipy.optimize import minimize_scalar

In [ ]:
p = len(z)
marginal_likelihood = CARMA._ind_Normal_fixed_sigma_marginal_external
tau_sample = tau
if outlier_switch:
    outlier_likelihood = CARMA._outlier_ind_Normal_marginal_external
    outlier_tau = tau

B = Max_Model_Dim
stored_bf = 0
Sigma = ld_matrix

S = []

null_model = ""
null_margin = CARMA._prior_dist(null_model, lambda_val=lambda_val, p=p)

B_list = pd.DataFrame({"set_gamma_margin": [null_margin], "matrix_gamma": [""]})

if input_conditional_S_list is None:
    conditional_S = []
else:
    conditional_S = input_conditional_S_list
    S = conditional_S


In [ ]:
for _i in range(0, inner_all_iter):
    for _j in range(0, 10):
        set_gamma = CARMA._set_gamma_func(
            input_S=S, p=p, condition_index=conditional_S
        )

        if conditional_S is None:
            working_S = S
        else:
            working_S = np.sort(np.setdiff1d(S, conditional_S)).astype(int)

        set_gamma_margin: list[Any] = [None, None, None]
        set_gamma_prior: list[Any] = [None, None, None]
        matrix_gamma: list[Any] = [None, None, None]

        for i in range(0, len(set_gamma)):
            if set_gamma[i] is not None:
                matrix_gamma[i] = CARMA._index_fun(set_gamma[i])
                p_S = set_gamma[i].shape[1]
                set_gamma_margin[i] = np.apply_along_axis(
                    marginal_likelihood,
                    1,
                    set_gamma[i] + 1,
                    Sigma=Sigma,
                    z=z,
                    tau=tau_sample,
                    p_S=p_S,
                )
                set_gamma_prior[i] = np.array(
                    [
                        CARMA._prior_dist(model, lambda_val=lambda_val, p=p)
                        for model in matrix_gamma[i]
                    ]
                )
                set_gamma_margin[i] = set_gamma_prior[i] + set_gamma_margin[i]
            else:
                set_gamma_margin[i] = np.array(null_margin)
                set_gamma_prior[i] = 0
                matrix_gamma[i] = np.array(null_model)

        columns = ["set_gamma_margin", "matrix_gamma"]
        add_B = pd.DataFrame(columns=columns)

        for i in range(len(set_gamma)):
            if isinstance(set_gamma_margin[i].tolist(), list):
                new_row = pd.DataFrame(
                    {
                        "set_gamma_margin": set_gamma_margin[i].tolist(),
                        "matrix_gamma": matrix_gamma[i].tolist(),
                    }
                )
                add_B = pd.concat([add_B, new_row], ignore_index=True)
            else:
                new_row = pd.DataFrame(
                    {
                        "set_gamma_margin": [set_gamma_margin[i].tolist()],
                        "matrix_gamma": [matrix_gamma[i].tolist()],
                    }
                )
                add_B = pd.concat([add_B, new_row], ignore_index=True)

        # Add visited models into the storage space of models
        B_list = pd.concat([B_list, add_B], ignore_index=True)
        B_list = B_list.drop_duplicates(
            subset="matrix_gamma", ignore_index=True
        )
        B_list = B_list.sort_values(
            by="set_gamma_margin", ignore_index=True, ascending=False
        )

        if len(working_S) == 0:
            # Create a DataFrame set.star
            set_star = pd.DataFrame(
                {
                    "set_index": [0, 1, 2],
                    "gamma_set_index": [np.nan, np.nan, np.nan],
                    "margin": [np.nan, np.nan, np.nan],
                }
            )

            # Assuming set.gamma.margin and current.log.margin are defined
            aa = set_gamma_margin[1]
            aa = aa - aa[np.argmax(aa)]

            min_half_len = min(len(aa), floor(p / 2))
            decr_ind = np.argsort(np.exp(aa))[::-1]
            decr_half_ind = decr_ind[:min_half_len]

            probs = np.exp(aa)[decr_half_ind]

            chosen_index = np.random.choice(
                decr_half_ind, 1, p=probs / np.sum(probs)
            )
            set_star.at[1, "gamma_set_index"] = chosen_index[0]
            set_star.at[1, "margin"] = set_gamma_margin[1][chosen_index[0]]

            S = set_gamma[1][chosen_index[0]].tolist()

        else:
            set_star = pd.DataFrame(
                {
                    "set_index": [0, 1, 2],
                    "gamma_set_index": [np.nan, np.nan, np.nan],
                    "margin": [np.nan, np.nan, np.nan],
                }
            )
            for i in range(0, 3):
                aa = set_gamma_margin[i]
                if np.size(aa) > 1:
                    aa = aa - aa[np.argmax(aa)]
                    chosen_index = np.random.choice(
                        range(0, np.size(set_gamma_margin[i])),
                        1,
                        p=np.exp(aa) / np.sum(np.exp(aa)),
                    )
                    set_star.at[i, "gamma_set_index"] = chosen_index
                    set_star.at[i, "margin"] = set_gamma_margin[i][chosen_index]
                else:
                    set_star.at[i, "gamma_set_index"] = 0
                    set_star.at[i, "margin"] = set_gamma_margin[i]

            if outlier_switch:
                for i in range(1, len(set_gamma)):
                    test_log_BF: float = 100
                    while True:
                        aa = set_gamma_margin[i]
                        aa = aa - aa[np.argmax(aa)]
                        chosen_index = np.random.choice(
                            range(0, np.size(set_gamma_margin[i])),
                            1,
                            p=np.exp(aa) / np.sum(np.exp(aa)),
                        )
                        set_star.at[i, "gamma_set_index"] = chosen_index
                        set_star.at[i, "margin"] = set_gamma_margin[i][
                            chosen_index
                        ]

                        test_S = set_gamma[i][int(chosen_index), :]

                        modi_Sigma = Sigma.copy()
                        if np.size(test_S) > 1:
                            modi_ld_S = modi_Sigma[test_S][:, test_S]

                            result = minimize_scalar(
                                CARMA._ridge_fun,
                                bounds=(0, 1),
                                args=(
                                    Sigma,
                                    modi_ld_S,
                                    test_S,
                                    z,
                                    outlier_tau,
                                    outlier_likelihood,
                                ),
                                method="bounded",
                            )
                            modi_ld_S = result.x * modi_ld_S + (
                                1 - result.x
                            ) * np.eye(len(modi_ld_S))

                            modi_Sigma[np.ix_(test_S, test_S)] = modi_ld_S

                            test_log_BF = outlier_likelihood(
                                test_S + 1, Sigma, z, outlier_tau, len(test_S)
                            ) - outlier_likelihood(
                                test_S + 1,
                                modi_Sigma,
                                z,
                                outlier_tau,
                                len(test_S),
                            )
                            test_log_BF = -np.abs(test_log_BF)

                        if np.exp(test_log_BF) < outlier_BF_index:
                            set_gamma[i] = np.delete(
                                set_gamma[i],
                                int(set_star["gamma_set_index"][i]),
                                axis=0,
                            )
                            set_gamma_margin[i] = np.delete(
                                set_gamma_margin[i],
                                int(set_star["gamma_set_index"][i]),
                                axis=0,
                            )
                            conditional_S = np.concatenate(
                                [conditional_S, np.setdiff1d(test_S, working_S)]
                            )
                            conditional_S = (
                                np.unique(conditional_S).astype(int).tolist()
                            )
                        else:
                            break

            if len(working_S) == num_causal:
                set_star = set_star.drop(1)
                aa = set_star["margin"] - max(set_star["margin"])
                sec_sample = np.random.choice(
                    [0, 2], 1, p=np.exp(aa) / np.sum(np.exp(aa))
                )
                ind_sec = int(
                    set_star["gamma_set_index"][
                        set_star["set_index"] == int(sec_sample)
                    ]
                )
                S = set_gamma[sec_sample[0]][ind_sec].tolist()
            else:
                aa = set_star["margin"] - max(set_star["margin"])
                sec_sample = np.random.choice(
                    range(0, 3), 1, p=np.exp(aa) / np.sum(np.exp(aa))
                )
                if set_gamma[sec_sample[0]] is not None:
                    S = set_gamma[sec_sample[0]][
                        int(set_star["gamma_set_index"][sec_sample[0]])
                    ].tolist()
                else:
                    sec_sample = np.random.choice(
                        range(1, 3),
                        1,
                        p=np.exp(aa)[[1, 2]] / np.sum(np.exp(aa)[[1, 2]]),
                    )
                    S = set_gamma[sec_sample[0]][
                        int(set_star["gamma_set_index"][sec_sample[0]])
                    ].tolist()

        for item in conditional_S:
            if item not in S:
                S.append(item)
    # END h_ind loop
    #
    if conditional_S is not None:
        all_c_index = []
        index_array = [s.split(",") for s in B_list["matrix_gamma"]]
        for tt in conditional_S:
            tt_str = str(tt)
            ind = [
                i for i, sublist in enumerate(index_array) if tt_str in sublist
            ]
            all_c_index.extend(ind)

        all_c_index = list(set(all_c_index))

        if len(all_c_index) > 0:
            temp_B_list = B_list.copy()
            temp_B_list = B_list.drop(all_c_index)
        else:
            temp_B_list = B_list.copy()
    else:
        temp_B_list = B_list.copy()

    result_B_list = temp_B_list[: min(int(B), len(temp_B_list))]

    rb1 = result_B_list["set_gamma_margin"]

    difference = abs(rb1[: (len(rb1) // 4)].mean() - stored_bf)

    if difference < epsilon:
        break
    else:
        stored_bf = rb1[: (len(rb1) // 4)].mean()

/var/folders/p5/4t9crp1563l792qz8xz_3x5h0000gq/T/ipykernel_25826/2604485643.py:138: DeprecationWarning:

Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)

/var/folders/p5/4t9crp1563l792qz8xz_3x5h0000gq/T/ipykernel_25826/2604485643.py:138: DeprecationWarning:

Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)

/var/folders/p5/4t9crp1563l792qz8xz_3x5h0000gq/T/ipykernel_25826/2604485643.py:138: DeprecationWarning:

Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)

/var/folders/p5/4t9crp1563l792qz8xz_3x5h0000gq/T/ipykernel_25826/2604485643.py:

ValueError: probabilities contain NaN

In [ ]:
aa = set_gamma_margin[i]

In [ ]:
B_list

,set_gamma_margin,matrix_gamma
0,624.933192,"128,288,370,378,396,422,452"
1,596.192342,"128,288,370,378,422,452"
2,581.693773,"128,288,370,396,422,452"
3,552.962296,"128,288,370,378,396,422"
4,469.727111,"128,370,378,396,422,452"
...,...,...
37777,NaN,"59,105,128,288,378,394,411,422,440,444"
37778,NaN,"59,105,128,288,378,396,411,422,440,444"
37779,NaN,"59,105,128,288,394,396,411,422,440,444"
37780,NaN,"59,105,288,378,394,396,411,422,440,444"


In [ ]:
Sigma.shape

(453, 453)

In [ ]:
p_S=10
CARMA._ind_Normal_fixed_sigma_marginal_external(index_vec_input=np.array([59,105,128,288,378,394,411,422,440,444]),Sigma=Sigma, 
                                                z=z, tau=0.3, p_S=p_S)


111.25914227692233

In [ ]:
tau=0.04
index_vec = np.array([59,105,128,288,378,394,411,422,440,444])
Sigma_S = Sigma[np.ix_(index_vec, index_vec)]
A = tau * np.eye(p_S)

det_S = det(Sigma_S + A)
Sigma_S_inv = inv(Sigma_S + A)

sub_z = z[index_vec]
zSigmaz_S = np.dot(sub_z.T, np.dot(Sigma_S_inv, sub_z))

In [ ]:
det(Sigma_S)

-0.18625511225147154

In [ ]:
det(Sigma_S+np.eye(p_S)*0.12)

0.031153872754756947

In [ ]:
det_S=0

In [ ]:
p_S / 2.0 * np.log(tau) - 0.5 * np.log(0.0000001) + zSigmaz_S / 2.0

315.83914962954367

In [ ]:
 marginal_likelihood,
                            1,
                            set_gamma[i] + 1,
                            Sigma=Sigma,
                            z=z,
                            tau=tau_sample,
                            p_S=p_S,

In [ ]:
p=np.exp(aa) / np.sum(np.exp(aa))

In [ ]:
np.sum(np.exp(aa))

nan

In [ ]:
aa1=aa

In [ ]:
aa=np.array([1,2,3,4,5,10,100])

In [ ]:
np.exp(aa) / np.sum(np.exp(aa))

array([1.01122149e-43, 2.74878501e-43, 7.47197234e-43, 2.03109266e-42,
       5.52108228e-42, 8.19401262e-40, 1.00000000e+00])

In [ ]:
aa_max = np.max(aa)
np.exp(aa - aa_max) / np.sum(np.exp(aa - aa_max))

array([1.01122149e-43, 2.74878501e-43, 7.47197234e-43, 2.03109266e-42,
       5.52108228e-42, 8.19401262e-40, 1.00000000e+00])

In [ ]:
aa = set_gamma_margin[i]
if np.size(aa) > 1:
    aa = aa - aa[np.argmax(aa)]
    chosen_index = np.random.choice(
        range(0, np.size(set_gamma_margin[i])),
        1,
        p=np.exp(aa) / np.sum(np.exp(aa)),
    )

In [ ]:
set_gamma_margin[i]

array([422.74097631, 423.34333589, 421.73875587, 421.78397104,
       422.89541685, 423.68921577, 424.84192749, 421.75042932,
       421.89269228, 421.56890184, 422.07472038, 422.14088769,
       422.23048325, 424.16344231, 421.45742109, 421.65904622,
       421.23245085, 421.19265572, 421.10034375, 422.11401519,
       421.13829497, 421.10983317, 421.16956932, 421.17078011,
       421.16554858, 421.17269427, 421.2547335 , 422.23672338,
       422.34381073, 421.25139562, 421.44002571, 422.83288733,
       422.90796517, 422.06148626, 424.10783571, 422.43373666,
       421.3218705 , 421.44158037, 421.52297081, 421.17226368,
       423.38589794, 422.01626027, 423.46014264, 421.01943923,
       421.47144506, 421.22382825, 422.95034708, 421.02049184,
       421.20694668, 421.02668455, 421.75735163, 421.6752135 ,
       421.76105936, 422.02747892, 424.27327274, 423.56910411,
       421.55696424, 422.70784762, 422.9470059 , 421.02057606,
       423.47539294, 421.54756595, 422.10402766, 421.06

In [ ]:
np.argmax(aa)

0

In [ ]:
np.exp(aa) / np.sum(np.exp(aa))

array([nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
       nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
       nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
       nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
       nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
       nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
       nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
       nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
       nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
       nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
       nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
       nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
       nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
       nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, na

In [ ]:
sl=StudyLocus.from_parquet(session,"gs://ukb_ppp_eur_data/collected")
df=sl.df

Py4JJavaError: An error occurred while calling o92.load.
: org.apache.hadoop.fs.UnsupportedFileSystemException: No FileSystem for scheme "gs"
	at org.apache.hadoop.fs.FileSystem.getFileSystemClass(FileSystem.java:3443)
	at org.apache.hadoop.fs.FileSystem.createFileSystem(FileSystem.java:3466)
	at org.apache.hadoop.fs.FileSystem.access$300(FileSystem.java:174)
	at org.apache.hadoop.fs.FileSystem$Cache.getInternal(FileSystem.java:3574)
	at org.apache.hadoop.fs.FileSystem$Cache.get(FileSystem.java:3521)
	at org.apache.hadoop.fs.FileSystem.get(FileSystem.java:540)
	at org.apache.hadoop.fs.Path.getFileSystem(Path.java:365)
	at org.apache.spark.sql.execution.datasources.DataSource$.$anonfun$checkAndGlobPathIfNecessary$1(DataSource.scala:752)
	at scala.collection.immutable.List.map(List.scala:293)
	at org.apache.spark.sql.execution.datasources.DataSource$.checkAndGlobPathIfNecessary(DataSource.scala:750)
	at org.apache.spark.sql.execution.datasources.DataSource.checkAndGlobPathIfNecessary(Dat

In [ ]:
df_df=df.select("studyLocusId").toPandas()

#df_df.iloc[3289]
#Out[42]: 
#studyLocusId    2530675590697900789
#Name: 3289, dtype: int64

#df_df.iloc[9340]
#Out[43]: 
#studyLocusId   -464208866647954250
#Name: 9340, dtype: int64

#df_df.iloc[12566]
#Out[44]: 
#studyLocusId   -5266354790453947393
#Name: 12566, dtype: int64

In [ ]:
df1= df.filter(f.col("studyLocusId") == -464208866647954250)

df1= df1.first()

In [ ]:
si=StudyIndex.from_parquet(session,"gs://ukb_ppp_eur_data/study_index")

In [ ]:
res=SusieFineMapperStep.susie_finemapper_one_studylocus_row_v3_dev_ss_gathered(
session=session,
study_locus_row=df1,
study_index=si,
radius= 1_000_000,
max_causal_snps=10,
susie_est_tausq=False,
run_carma=False,
run_sumstat_imputation=False,
carma_time_limit=600,
imputed_r2_threshold=0.8,
ld_score_threshold=4,
sum_pips=0.99,
primary_signal_pval_threshold=1,
secondary_signal_pval_threshold=1,
purity_min_r2_threshold=0,
purity_mean_r2_threshold=0,
cs_lbf_thr=2,
)

print(res["study_locus"].df.show())
print(res["log"])


2024-06-21 22:21:23.485 Hail: INFO: Table.join: renamed the following fields on the right to avoid name conflicts:
    'freq_index_dict' -> 'freq_index_dict_1'
    'rf' -> 'rf_1'
    'faf_index_dict' -> 'faf_index_dict_1'
    'popmax_index_dict' -> 'popmax_index_dict_1'
    'age_index_dict' -> 'age_index_dict_1'
    'age_distribution' -> 'age_distribution_1'
    'freq_meta' -> 'freq_meta_1'
2024-06-21 22:22:05.354 Hail: INFO: Ordering unsorted dataset with network shuffle
2024-06-21 22:22:44.090 Hail: INFO: Coerced sorted dataset
----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 63775)
Traceback (most recent call last):
  File "/Users/yt4/.pyenv/versions/3.10.8/lib/python3.10/socketserver.py", line 316, in _handle_request_noblock
    self.process_request(request, client_address)
  File "/Users/yt4/.pyenv/versions/3.10.8/lib/python3.10/socketserver.py", line 347, in process_request
    self.finish_request(request, client_address)

+----------------+--------------------+--------------------+-------------------+--------------------+----------------+----------+--------+-----------------+------------------+------------------+-------------------+-------------------+--------------+--------------+
|credibleSetIndex|        studyLocusId|             studyId|             region|               locus|       variantId|chromosome|position|finemappingMethod|credibleSetlog10BF|      purityMeanR2|        purityMinR2|             zScore|pValueMantissa|pValueExponent|
+----------------+--------------------+--------------------+-------------------+--------------------+----------------+----------+--------+-----------------+------------------+------------------+-------------------+-------------------+--------------+--------------+
|               1|-6547045969040230899|UKB_PPP_EUR_HBEGF...|X:64902662-66902662|[{X_65011724_G_T,...|  X_65011724_G_T|         X|65011724|        SuSiE-inf|12.221549455802414|0.8120673857155859|0.429124879

In [ ]:
x=session.spark.read.parquet("gs://finngen_data/100_3mb_carma/finemapped/*")

In [ ]:
from pyspark.sql.functions import split
from pyspark.sql.types import IntegerType
unique_df = x.select("region").distinct()
df = unique_df.withColumn("region_split", split(x["region"], "[:-]"))

df = df.withColumn("column1", df["region_split"].getItem(0))
df = df.withColumn("column2", df["region_split"].getItem(1))
df = df.withColumn("column3", df["region_split"].getItem(2))
df = df.withColumn("column4", (((f.col("column3") + f.col("column2")) / 2)).cast(IntegerType()))

In [ ]:
df.show()

+--------------------+--------------------+-------+---------+---------+---------+
|              region|        region_split|column1|  column2|  column3|  column4|
+--------------------+--------------------+-------+---------+---------+---------+
|12:99130939-10213...|[12, 99130939, 10...|     12| 99130939|102130939|100630939|
|2:134768345-13776...|[2, 134768345, 13...|      2|134768345|137768345|136268345|
|12:26306268-29306268|[12, 26306268, 29...|     12| 26306268| 29306268| 27806268|
|1:168599357-17159...|[1, 168599357, 17...|      1|168599357|171599357|170099357|
|6:167759955-17075...|[6, 167759955, 17...|      6|167759955|170759955|169259955|
| 6:26949375-29949375|[6, 26949375, 299...|      6| 26949375| 29949375| 28449375|
|19:31838930-34838930|[19, 31838930, 34...|     19| 31838930| 34838930| 33338930|
| 9:20525494-23525494|[9, 20525494, 235...|      9| 20525494| 23525494| 22025494|
|6:149183634-15218...|[6, 149183634, 15...|      6|149183634|152183634|150683634|
|11:29274440-322

In [ ]:
si=StudyIndex.from_parquet(session,"gs://finngen_data/r10/study_index")

In [ ]:
sl=StudyLocus.from_parquet(session,"gs://genetics-portal-dev-analysis/dc16/finngen_100_sl.parquet_PATCHED-2024-05-30")

In [ ]:
sl_df=sl.df.select("position").toPandas()

In [ ]:
df_df=df.toPandas()

In [ ]:
~sl_df["position"].isin(df_df["column4"])
# Convert the Series to a list
bool_list = (~sl_df["position"].isin(df_df["column4"])).tolist()

print(bool_list)

[True, False, False, False, False, False, False, False, False, True, False, False, True, False, False, False, False, False, False, False, False, False, False, False, False, True, False, False, False, False, True, False, False, True, False, True, True, False, True, True, False, False, False, False, False, False, False, False, False, False, True, True, False, True, False, True, False, False, False, False, True, False, False, True, True, False, True, False, False, False, True, False, True, True, False, False, True, False, True, False, True, True, False, False, False, False, False, False, False, True, True, False, True, True, False, False, True, False, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, T

In [ ]:
not_in_df = sl.df.filter(~f.col('position').isin(df.select('column4').rdd.flatMap(lambda x: x).collect()))

not_in_df.show()

+--------------------+------------------+----------+---------+------+--------------------+----------+------+--------------+--------------+-------------------------------+-------------+-------------------+---------------+-----------------+----------------+------------------+------------+-----------+----------+--------+----------+-----+--------------------+
|        studyLocusId|         variantId|chromosome| position|region|             studyId|      beta|zScore|pValueMantissa|pValueExponent|effectAlleleFrequencyFromSource|standardError|subStudyDescription|qualityControls|finemappingMethod|credibleSetIndex|credibleSetlog10BF|purityMeanR2|purityMinR2|locusStart|locusEnd|sampleSize|ldSet|               locus|
+--------------------+------------------+----------+---------+------+--------------------+----------+------+--------------+--------------+-------------------------------+-------------+-------------------+---------------+-----------------+----------------+------------------+----------

In [ ]:
sl.df.count()

2850

In [ ]:
df.count()

69